En nuestra propuesta de proyecto integrador mencionamos el uso de Apache Kafka o AWS Kinesis para la ingesta masiva de eventos en tiempo real (modelo productor-suscriptor), acoplado a un procesamiento Spark Structured Streaming. Por las limitaciones de la versión gratuita de Databricks, los datos base son los CSV estáticos de Kaggle depositados en un Volume (la descarga ocurre en `01_ingesta_kaggle`, fuera de este notebook).

Estrategia Kappa (replay de streaming sobre archivos estáticos):
Para mantener la coherencia con la propuesta de "Arquitectura de Datos de Alto Rendimiento", la capa Bronze **no se ingesta en batch**: se construye con **Auto Loader (`cloudFiles`) + `Trigger.AvailableNow` + `checkpoint`**, que procesa los CSV estáticos **como si fueran un flujo continuo** y luego se detiene (no corre en bucle). Así el código es de **streaming real** —no batch disfrazado—, demostrando el patrón Kappa sin broker ni AWS. El `checkpoint` hace la operación idempotente (no reprocesa archivos ya ingeridos); para reconstruir desde cero la celda incluye un bloque de RESET. Si Auto Loader no estuviera disponible en el entorno, queda un *fallback* batch comentado al final de la celda.

Diseño de la Arquitectura Medallion (Bronze, Silver, Gold):
- El objetivo de esta arquitectura es refinar iterativamente los datos, garantizando calidad, linaje y rendimiento. Como estándar en Databricks, no usaremos simplemente archivos Parquet, sino Delta Tables (que están basadas en Parquet pero añaden la capa transaccional ACID)

El siguiente código lee el archivo CSV que descargamos desde Kaggle y fue almacenado en la carpeta e-commerce_raw en el notebook `01_ingesta_kaggle`

In [0]:
#programa para leer los archivos en la carpeta e-commerce

import os

ruta = '/Volumes/workspace/default/ecommerce_raw'

archivos = os.listdir(ruta)

print("="*31)
print("LECTURA DE ARCHIVOS ALMACENADOS\nEN CARPETA ecommerce_raw")
print("="*31)


for f in archivos:
    size = os.path.getsize(f'{ruta}/{f}')
    print(f"📄 {f}  →  {size/1e9:.2f} GB")

print()

# Capa Bronze
Objetivo: Almacenar los datos tal cual llegaron desde Kaggle (inmutabilidad), sirviendo como fuente de la verdad para reprocesamientos.
- Qué hacemos aquí?: Leer los CSV del volumen /Volumes/workspace/default/ecommerce_raw y escribirlos en formato Delta sin alterar esquemas ni limpiar datos.

- Por qué?: Si en el futuro descubrimos que limpiamos mal una variable en la capa Silver, siempre podemos volver a la Bronze sin tener que re-descargar de Kaggle.

In [0]:
# 01_Ingestion_Bronze — KAPPA: replay de streaming con Auto Loader (cloudFiles) + Trigger.AvailableNow.
# Sustituye la lectura batch (queda como FALLBACK comentado al final por si Free Edition no trae Auto Loader).
from pyspark.sql.types import StructType, StructField, StringType, FloatType
from pyspark.sql.functions import current_timestamp, col, to_date, to_timestamp

# 1. Volumen
print("1. Verificando/Creando Volumen 'e_commerce'...")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.e_commerce")

# 2. Esquema estricto (mismo de siempre)
schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("user_id", StringType(), True),
    StructField("user_session", StringType(), True),
])

# 3. Rutas
raw_dir           = "/Volumes/workspace/default/ecommerce_raw/"               # landing de los CSV (los baja 01_ingesta)
bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"
checkpoint_path   = "/Volumes/workspace/default/e_commerce/bronze/_checkpoint"
schema_path       = "/Volumes/workspace/default/e_commerce/bronze/_schema"

# 3.b RESET — al convertir de batch a streaming hay que reconstruir el Bronze desde cero: si no, el append
#     del stream DUPLICARIA sobre el Bronze batch ya existente. Deja esto activo la 1a vez. Si despues quieres
#     solo ingestar archivos NUEVOS (replay incremental real, Kappa), comenta este bloque: el checkpoint ya
#     recuerda lo procesado y no reprocesa lo viejo.
for _path in (bronze_table_path, checkpoint_path, schema_path):
    try:
        dbutils.fs.rm(_path, True)
    except Exception:
        pass

# 4. Lectura como STREAM (Auto Loader). Trigger.AvailableNow = procesa lo disponible y PARA (no en bucle, doc 02 §4).
print("2. Leyendo Raw como stream (Auto Loader / cloudFiles)...")
df_raw = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .option("pathGlobFilter", "*.csv")          # ignora kaggle.json u otros no-CSV del landing
    .schema(schema)
    .load(raw_dir))

# 5. Linaje + particion por fecha de evento (identico al batch; doc 02 §3)
df_bronze = (df_raw
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("event_date", to_date(to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'"))))

# 6. Escritura como STREAM con checkpoint (idempotencia de streaming: el checkpoint evita reprocesar archivos)
print("3. Escribiendo Bronze (writeStream, availableNow)... esto tarda unos minutos.")
q = (df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .partitionBy("event_date")
    .trigger(availableNow=True)
    .outputMode("append")
    .start(bronze_table_path))
q.awaitTermination()

_n = spark.read.format("delta").load(bronze_table_path).count()
print(f"✅ Bronze (Kappa replay) listo: {_n:,} filas. Operacion repetible (checkpoint).")

# --- FALLBACK BATCH (descomentar SOLO si Auto Loader no estuviera disponible en el entorno) ---
# df_raw_b = spark.read.format("csv").option("header", "true").schema(schema).load(raw_dir + "*.csv")
# df_bronze_b = (df_raw_b
#     .withColumn("_ingest_timestamp", current_timestamp())
#     .withColumn("_source_file", col("_metadata.file_path"))
#     .withColumn("event_date", to_date(to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'"))))
# (df_bronze_b.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
#     .partitionBy("event_date").save(bronze_table_path))
# print("Bronze batch (fallback) listo.")

# Capa Silver

1. Tipado de Datos (Casting)

El código toma columnas que llegaron como texto plano y las convierte a sus tipos matemáticos correctos. La fecha de evento se convierte a un formato de tiempo continuo (t), y el precio pasa a ser una variable numérica continua (FloatType). Esto es vital, ya que los algoritmos matemáticos no pueden calcular recencias ni promedios sobre cadenas de texto.

2. Feature Engineering Temporal:

A partir de la marca de tiempo original (t), el motor distribuido extrajo características cíclicas y discretas:

- La fecha exacta (date).

- La hora del día (0,23).

- El día de la semana (1,7) y su nombre.

Esto permite que, en el EDA y modelado posterior, podamos encontrar patrones de propensión basados en ventanas horarias (por ejemplo, si la franja matutina tiene mayor intención de compra).

3. Aplanamiento Taxonómico Seguro (Parsing):

La columna category_code venía anidada (ej. electronics.smartphone.apple). El código utiliza una división por arreglos (arrays) e implementa una validación lógica con size() para extraer de manera segura la macro-categoría, sub-categoría y tipo de ítem. Si a un evento le faltan niveles taxonómicos, el código lo maneja sin colapsar, asignando un valor nulo de forma controlada.

4. Aseguramiento de Calidad (Data Quality & Imputation)

Aplicamos el principio de que la ausencia de información también es información.

* Imputación: Llenamos los valores faltantes en categorías y marcas con "Unknown", preservando el evento en lugar de borrarlo, lo cual es fundamental para los árboles de decisión.

* Limpieza de Ruido: Eliminamos (dropna) cualquier evento que no tuviera user_session o user_id. Un log sin sesión es un evento "huérfano" que matemáticamente no puede unirse a un customer journey, por lo que carece de valor predictivo.

5. Escritura Idempotente:

Finalmente, el script utiliza .mode("overwrite"). Esto garantiza que, sin importar cuántas veces ejecuten el notebook, la tabla resultante siempre representará el estado correcto y limpio de los datos, eliminando el riesgo de duplicidad masiva.

6. Limpieza — pasada 1 (4-jun, §2.3.1 paso 3):

Se añadieron dos pasos de limpieza **transferidos del EDA oficial** (`analysis/eda_ecommerce.ipynb`, sección 2):

* **`dropDuplicates(business_cols)`** — elimina filas de evento exactamente duplicadas (~0.06% en la muestra del EDA). Se deduplica por las 9 columnas de negocio, ignorando los metadatos de linaje (`_ingest_timestamp`, `_source_file`).
* **`filter(price > 0)`** — un precio 0/nulo no representa un evento de compra real y distorsiona la conversión.

> **Calidad revisada en pasada 2 (4-jun · doc 00 §17.5):** outliers de precio (auditado: precio acotado ~2574, sin outliers -> no se winsoriza) y sesiones-bot (auditado: cola despreciable 0.002% -> sin flag). Categoricas "Unknown" no backfilleable -> se mantiene + category_id para ML.

In [0]:
from pyspark.sql.functions import col, split, to_timestamp, to_date, hour, dayofweek, date_format, size, when, lit

bronze_table_path = "/Volumes/workspace/default/e_commerce/bronze/clickstream"
silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"

# Columnas de negocio (sin metadatos de linaje) para deduplicar filas EXACTAS
business_cols = ["event_time", "event_type", "product_id", "category_id",
                 "category_code", "brand", "price", "user_id", "user_session"]

df_bronze = spark.read.format("delta").load(bronze_table_path)

# --- Pasada 1 de limpieza (justificada por el EDA oficial, sección 2) ---
#   - dropDuplicates(business_cols): elimina filas de evento exactamente duplicadas (~0.06% en el EDA)
#   - filter(price > 0): un precio 0/nulo no representa un evento válido y distorsiona la conversión
# CALIDAD (pasada 2, revisado 4-jun -- decisiones registradas en doc 00 seccion 17.5):
#   - OUTLIERS de precio: auditado (p999=2562, max=2574; todas las macro topan ~2574 ->
#     precio acotado en la fuente, probable cap REES46). Con price>0 ya filtrado NO se winsoriza.
#   - SESIONES-BOT: auditado (p999=76 eventos/sesion; >200 ev = 506 sesiones = 0.002%). Cola
#     despreciable -> NO se anade flag (revisado, no amerita accion).
#   - CATEGORICAS 'Unknown': brand ~14%, macro ~32%; NO backfilleable desde category_id
#     (0 de 413). Se mantiene 'Unknown' como categoria; para ML se usa category_id (limpio, 690).
df_silver = df_bronze \
    .dropDuplicates(business_cols) \
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss 'UTC'")) \
    .withColumn("price", col("price").cast("float")) \
    .filter(col("price") > 0) \
    .withColumn("date", to_date(col("event_time"))) \
    .withColumn("hour", hour(col("event_time"))) \
    .withColumn("day_of_week_num", dayofweek(col("event_time"))) \
    .withColumn("day_name", date_format(col("event_time"), "EEEE")) \
    .withColumn("cat_array", split(col("category_code"), r"\.")) \
    .withColumn("macro_category", col("cat_array").getItem(0)) \
    .withColumn("sub_category", when(size(col("cat_array")) > 1, col("cat_array").getItem(1)).otherwise(lit("Unknown"))) \
    .withColumn("item_type", when(size(col("cat_array")) > 2, col("cat_array").getItem(2)).otherwise(lit("Unknown"))) \
    .drop("cat_array", "category_code") \
    .fillna({
        "brand": "Unknown",
        "macro_category": "Unknown"
    }) \
    .dropna(subset=["user_session", "user_id"])

# Guardar en Silver (idempotente)
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("date")               # doc 02 §3: Silver por fecha (queries por ventana temporal)
    .save(silver_table_path))

print("✅ Capa Silver: dedup + tipado + FE temporal + parsing taxonómico + price>0. Idempotente.")

# OPTIMIZE + ZORDER (doc 02 §3 regla 3-4): Silver particionada por fecha; ZORDER por category_id
# (columna selectiva de filtro frecuente). Se mide para sustentar con evidencia, no por intuicion.
spark.sql(f"OPTIMIZE delta.`{silver_table_path}` ZORDER BY (category_id)")
_d = spark.sql(f"DESCRIBE DETAIL delta.`{silver_table_path}`").select("partitionColumns","numFiles","sizeInBytes").first()
print(f"Silver -> particion={_d['partitionColumns']} | archivos={_d['numFiles']} | tamano={_d['sizeInBytes']/1e9:.2f} GB")


Validación rápida de escritura y datos en la capa Silver

In [0]:
# Lectura rápida de la capa Silver para validación
silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
df_silver_qa = spark.read.format("delta").load(silver_table_path)

# Seleccionamos las columnas nuevas que creamos y mostramos 10 registros donde sí había categoría
print("🔍 Validando Feature Engineering Temporal y de Taxonomía:")
df_silver_qa.select(
    "event_time", "date", "hour", "day_name", 
    "macro_category", "sub_category", "item_type"
).filter(col("macro_category") != "Unknown").show(10, truncate=False)

# Validar el esquema para asegurar los tipos de datos (ej. price como float)
df_silver_qa.printSchema()

> ℹ️ **EDA-funnel trasladado fuera del pipeline (4-jun).**
>
> Aquí vivía un *funnel global* (vista → carrito → compra) calculado sobre toda la capa Silver, con su interpretación. Se **retiró del pipeline** por dos razones (doc 00 §12; doc 02 §4):
> 1. **Separación Medallion ↔ EDA:** el pipeline debe ser determinista e idempotente; el análisis vive en `notebooks/analysis/`.
> 2. **Cuota Databricks Free:** ese funnel hacía `count()` + `collect()` sobre los 14 GB, re-escaneando todo el dataset solo para un gráfico.
>
> El **funnel oficial del PI** (mejor construido, por unidad = producto-en-sesión, con corrección del muestreo) está en **`notebooks/analysis/eda_ecommerce.ipynb §4.1`**. Este notebook queda **solo con Medallion**: Bronze → Silver → Gold.

# Capa Gold


El siguiente paso en el flujo de trabajo CRISP-DM que planteamos es la Preparación de los Datos para Modelado. Esto significa que debemos construir la Capa Gold.

Mientras que Silver vive a nivel de evento (millones de clics), Gold debe vivir al nivel de nuestra unidad de predicción. Como nuestro objetivo es predecir la propensión basada en la recencia de clics y profundidad de navegación para manejar el desbalanceo extremo del 97% de no-compras, nuestra capa Gold será una Matriz de Características (Feature Store) donde:

- 1 fila = 1 sesión de usuario.

- Columnas (X) = Características de navegación (antes del corte temporal).

- Columna Objetivo (Y) = Variable binaria (1 si compró, 0 si no compró).

Esta es la matriz tabular masiva que consumirán directamente tus algoritmos. Ej: XGBoost o LightGBM.

In [0]:
from pyspark.sql import functions as F

silver_table_path = "/Volumes/workspace/default/e_commerce/silver/clickstream_clean"
gold_table_path   = "/Volumes/workspace/default/e_commerce/gold/features_session"

df_silver = spark.read.format("delta").load(silver_table_path)

# 1. Corte anti-fuga DETERMINISTA: primer evento critico (cart/purchase) por sesion; nos quedamos
#    con lo ESTRICTAMENTE anterior (event_time < t_crit). Evita el gotcha del frame RANGE.
t_crit = (df_silver.filter(F.col("event_type").isin("cart", "purchase"))
          .groupBy("user_session").agg(F.min("event_time").alias("t_crit")))

df_pre = (df_silver.join(t_crit, on="user_session", how="left")
          .filter(F.col("t_crit").isNull() | (F.col("event_time") < F.col("t_crit"))))

# 2. Universo de sesiones + target + user_id + ancla temporal.
#    session_start_ts = primer evento de la sesion = inicio, previo a cart/purchase (anti-fuga safe).
# user_id DETERMINISTA (auditoria 5-jun): en sesiones multi-usuario toma el user_id del PRIMER evento
# (event_time asc; user_id como desempate); session_start_ts sale del mismo struct. Reemplaza F.first() sin orden.
df_session = (df_silver.groupBy("user_session").agg(
    F.max(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("target_purchase"),
    F.min(F.struct(F.col("event_time"), F.col("user_id"))).alias("_first_evt"))
    .withColumn("user_id", F.col("_first_evt.user_id"))
    .withColumn("session_start_ts", F.col("_first_evt.event_time"))
    .drop("_first_evt"))

# 3. Features conductuales PRE-corte (1 fila/sesion). Set enriquecido en la pasada 2 (EDA seccion 17.1):
#    - categoricas: el string macro_category trae ~32% 'Unknown' NO backfilleable (category_id es
#      limpio: 690 valores, 0 ambiguos) -> se anade categories_explored_cid con category_id;
#      categories_explored (macro) se conserva por compatibilidad del contrato.
#    - senal de precio (no existia en Gold) y de negocio (electronica = 75% del revenue).
df_features = df_pre.groupBy("user_session").agg(
    F.count("product_id").alias("total_views"),
    F.countDistinct("product_id").alias("distinct_products_viewed"),
    F.countDistinct("brand").alias("brands_compared"),
    F.countDistinct("macro_category").alias("categories_explored"),
    F.countDistinct("category_id").alias("categories_explored_cid"),
    (F.unix_timestamp(F.max("event_time")) - F.unix_timestamp(F.min("event_time"))).alias("browsing_duration_sec"),
    F.round(F.avg("price"), 2).alias("avg_price_viewed"),
    F.round(F.max("price"), 2).alias("max_price_viewed"),
    F.round(F.avg((F.col("macro_category") == "electronics").cast("int")), 4).alias("electronics_view_share"))

# 4. JOIN LEFT desde el universo (no se pierde ningun positivo) + banderas temporales y features derivadas.
df_gold = (df_session
           .join(df_features, on="user_session", how="left")
           .withColumn("sin_navegacion_previa", F.col("total_views").isNull())
           .fillna({"total_views": 0, "distinct_products_viewed": 0,
                    "brands_compared": 0, "categories_explored": 0,
                    "categories_explored_cid": 0, "browsing_duration_sec": 0,
                    "avg_price_viewed": 0.0, "max_price_viewed": 0.0,
                    "electronics_view_share": 0.0})
           # --- ancla temporal y banderas ---
           .withColumn("session_date", F.to_date("session_start_ts"))
           .withColumn("session_hour", F.hour("session_start_ts"))
           .withColumn("day_of_week",  F.dayofweek("session_start_ts"))
           .withColumn("is_weekend", F.col("day_of_week").isin(1, 7).cast("int"))
           .withColumn("is_black_friday",
                       (F.col("session_date") == F.to_date(F.lit("2019-11-29"))).cast("int"))
           # label_window_corrupt: ventana 14-17 nov corrupta por ETL (criterio: etiqueta rota O volumen de eventos anomalo). 15=0 compras, 16-17 volcado del backlog; 14 = volcado de eventos: views/carts por encima de Black Friday con compras planas.
           #   Se MARCA, no se borra: cuarentena = filtro de modelado/agregacion (EDA 6.1.bis inv. 1).
           .withColumn("label_window_corrupt",
                       F.col("session_date").between(F.to_date(F.lit("2019-11-14")),
                                                     F.to_date(F.lit("2019-11-17"))).cast("int"))
           # --- decisividad / ritmo (EDA: el comprador decide rapido y navega menos) ---
           # revisit_intensity decorrelaciona el par total_views ~ distinct_products (corr 0.92).
           .withColumn("revisit_intensity",
                       F.round(F.when(F.col("distinct_products_viewed") > 0,
                               F.col("total_views") / F.col("distinct_products_viewed")).otherwise(F.lit(0.0)), 3))
           .withColumn("views_per_minute",
                       F.round(F.when(F.col("browsing_duration_sec") > 0,
                               F.col("total_views") / (F.col("browsing_duration_sec") / 60.0)).otherwise(F.lit(0.0)), 3))
           .withColumn("avg_inter_event_sec",
                       F.round(F.when(F.col("total_views") > 0,
                               F.col("browsing_duration_sec") / F.col("total_views")).otherwise(F.lit(0.0)), 2))
           .drop("session_start_ts"))

# 5. Guardamos la Capa Gold
print("Escribiendo Capa Gold (Matriz de Caracteristicas)...")
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_table_path))

print("OK Capa Gold pasada 2 (features): + categories_explored_cid, avg/max_price_viewed, "
      "electronics_view_share, is_weekend, revisit_intensity, views_per_minute, avg_inter_event_sec.")


# OPTIMIZE + ZORDER (doc 02 §3): la Gold por sesion es pequena (<< 1TB) -> NO se particiona
# (particionar 23M filas por fecha crearia ~25MB/particion = anti-patron de archivos pequenos).
# ZORDER por session_date (poda el split train/test via data-skipping) y user_id (clustering/segmentos).
spark.sql(f"OPTIMIZE delta.`{gold_table_path}` ZORDER BY (session_date, user_id)")
_d = spark.sql(f"DESCRIBE DETAIL delta.`{gold_table_path}`").select("partitionColumns","numFiles","sizeInBytes").first()
print(f"Gold -> particion={_d['partitionColumns']} (vacio = por diseno) | archivos={_d['numFiles']} | tamano={_d['sizeInBytes']/1e9:.2f} GB")


## ✅ Validación de la Gold corregida (§2.3.1 paso 3)

Verifica el **grano (1 fila = 1 sesión)**, la **tasa de etiqueta** y las sesiones preservadas, sobre la Gold recién escrita. El diagnóstico completo del `join` (antes/después, sobre muestra, con el mecanismo del sesgo) vive en **`02b_diagnostico_gold_join.ipynb`**.

In [0]:
# QA de la Gold corregida: grano (1 fila = 1 sesión) y tasa de etiqueta, sobre la Gold ya escrita.
from pyspark.sql import functions as F

gold = spark.read.format("delta").load(gold_table_path)

n_rows = gold.count()
n_sess = gold.select("user_session").distinct().count()
n_dup  = gold.groupBy("user_session").count().filter(F.col("count") > 1).count()
tasa   = gold.agg(F.avg("target_purchase")).first()[0]
n_flag = gold.filter(F.col("sin_navegacion_previa")).count()

print(f"Filas Gold...................: {n_rows:,}")
print(f"Sesiones distintas...........: {n_sess:,}")
print(f"Grano 1 fila = 1 sesión......: {n_rows == n_sess}   (duplicados: {n_dup})")
print(f"Tasa de etiqueta (purchase)..: {tasa:.4f}")
print(f"Sesiones sin navegación previa: {n_flag:,}  (entran con features=0 + flag; ya no se descartan)")

assert n_rows == n_sess and n_dup == 0, "GRANO ROTO: hay más filas que sesiones"
print("✅ Gold válida: grano correcto (1 fila = 1 sesión) y sin pérdida de positivos por el join.")